In [1]:
import numpy as np 
import pandas  as pd

**Dataset Loading**

In [2]:
match = pd.read_csv('matches.csv')
delivery= pd.read_csv('deliveries.csv')

**Data Inspection**

In [3]:
match.sample(4)

,id,Season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
70,71,IPL-2008,Chennai,26-04-2008,Kolkata Knight Riders,Chennai Super Kings,Kolkata Knight Riders,bat,normal,0,Chennai Super Kings,0,9,JDP Oram,"MA Chidambaram Stadium, Chepauk",BF Bowden,AV Jayaprakash,NaN
9,10,IPL-2017,Mumbai,12-04-2017,Sunrisers Hyderabad,Mumbai Indians,Mumbai Indians,field,normal,0,Mumbai Indians,0,4,JJ Bumrah,Wankhede Stadium,Nitin Menon,CK Nandan,NaN
731,11327,IPL-2019,Jaipur,20-04-2019,Mumbai Indians,Rajasthan Royals,Rajasthan Royals,field,normal,0,Rajasthan Royals,0,5,SPD Smith,Sawai Mansingh Stadium,S Ravi,Yeshwant Barde,O Nandan
680,7938,IPL-2018,Delhi,12-05-2018,Delhi Daredevils,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Royal Challengers Bangalore,0,5,AB de Villiers,Feroz Shah Kotla,Kumar Dharmasena,Anil Chaudhary,K Ananthapadmanabhan


In [4]:
delivery.head(4)

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN


In [5]:
match.isnull().sum()

id                   0
Season               0
city                 7
date                 0
team1                0
team2                0
toss_winner          0
toss_decision        0
result               0
dl_applied           0
winner               4
win_by_runs          0
win_by_wickets       0
player_of_match      4
venue                0
umpire1              2
umpire2              2
umpire3            637
dtype: int64

In [6]:
delivery.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    170244
dismissal_kind      170244
fielder             172630
dtype: int64

In [7]:
match.shape

(756, 18)

In [8]:
delivery.shape

(179078, 21)

**Removing unwanted features**

In [9]:
match = match.loc[:,['id','city','team1', 'team2','winner','toss_winner','toss_decision','result','win_by_runs','dl_applied','win_by_wickets','player_of_match','venue']]

In [10]:
delivery = delivery.loc[:,['match_id','inning','batting_team','bowling_team', 'over','ball','batsman','non_striker','bowler','is_super_over','bye_runs','legbye_runs','noball_runs','penalty_runs','batsman_runs','extra_runs','total_runs','player_dismissed','dismissal_kind','fielder']]

In [11]:
delivery.groupby(['match_id','inning']).sum()['total_runs']

match_id  inning
1         1         207
          2         172
2         1         184
          2         187
3         1         183
                   ... 
11413     2         170
11414     1         155
          2         162
11415     1         152
          2         157
Name: total_runs, Length: 1528, dtype: int64

In [12]:
## conversion to dataframe
total_score_df = delivery.groupby(['match_id','inning']).sum()['total_runs'].reset_index()

In [13]:
total_score_df=total_score_df[total_score_df['inning'] == 1]

**merging total_score_df to match on the basis of id and match_id**

In [14]:
match = match.merge(total_score_df[['match_id','total_runs']],left_on='id',right_on='match_id')

In [15]:
match.columns

Index(['id', 'city', 'team1', 'team2', 'winner', 'toss_winner',
       'toss_decision', 'result', 'win_by_runs', 'dl_applied',
       'win_by_wickets', 'player_of_match', 'venue', 'match_id', 'total_runs'],
      dtype='object')

**Preprocessing**

In [16]:
match['team1'].unique()

array(['Sunrisers Hyderabad', 'Mumbai Indians', 'Gujarat Lions',
       'Rising Pune Supergiant', 'Royal Challengers Bangalore',
       'Kolkata Knight Riders', 'Delhi Daredevils', 'Kings XI Punjab',
       'Chennai Super Kings', 'Rajasthan Royals', 'Deccan Chargers',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Rising Pune Supergiants',
       'Delhi Capitals'], dtype=object)

In [17]:
#### conversion Delhi Daredevils --> Delhi Capitals 
#### and Deccan Chargers --> Sunrisers Hyderabad

match['team1']=match['team1'].str.replace('Delhi Daredevils', 'Delhi Capitals')
match['team1']=match['team1'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
match['team2']=match['team2'].str.replace('Delhi Daredevils', 'Delhi Capitals')
match['team2']=match['team2'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')


In [18]:
match['team1'].unique()

array(['Sunrisers Hyderabad', 'Mumbai Indians', 'Gujarat Lions',
       'Rising Pune Supergiant', 'Royal Challengers Bangalore',
       'Kolkata Knight Riders', 'Delhi Capitals', 'Kings XI Punjab',
       'Chennai Super Kings', 'Rajasthan Royals', 'Kochi Tuskers Kerala',
       'Pune Warriors', 'Rising Pune Supergiants'], dtype=object)

In [19]:
#### since some of teams are not existing in present time so we have to remove them
teams = ['Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bangalore',
       'Kolkata Knight Riders', 'Kings XI Punjab',
       'Chennai Super Kings', 'Rajasthan Royals',
       'Delhi Capitals']



match = match[match['team1'].isin(teams)]
match = match[match['team2'].isin(teams)]
match = match[match['winner'].isin(teams)]
delivery = delivery[delivery['bowling_team'].isin(teams)]
delivery = delivery[delivery['batting_team'].isin(teams)]

In [20]:
match['dl_applied'].value_counts()   ### 0 = no rain, 1 = rain

dl_applied
0    542
1     13
Name: count, dtype: int64

In [21]:

match = match[match['dl_applied']==0]
match['dl_applied'].value_counts()

dl_applied
0    542
Name: count, dtype: int64

In [22]:
match = match.loc[:,['match_id','city','winner','total_runs','toss_decision']]

In [23]:
delivery = match.merge(delivery,on='match_id')

In [24]:
delivery = delivery[delivery['inning']==2]

In [25]:
delivery.head(5)

,match_id,city,winner,total_runs_x,toss_decision,inning,batting_team,bowling_team,over,ball,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder
125,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,...,0,0,0,0,1,0,1,NaN,NaN,NaN
126,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,...,0,0,0,0,0,0,0,NaN,NaN,NaN
127,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,...,0,0,0,0,0,0,0,NaN,NaN,NaN
128,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,...,0,0,0,0,2,0,2,NaN,NaN,NaN
129,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,...,0,0,0,0,4,0,4,NaN,NaN,NaN


In [26]:
delivery.columns

Index(['match_id', 'city', 'winner', 'total_runs_x', 'toss_decision', 'inning',
       'batting_team', 'bowling_team', 'over', 'ball', 'batsman',
       'non_striker', 'bowler', 'is_super_over', 'bye_runs', 'legbye_runs',
       'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs',
       'total_runs_y', 'player_dismissed', 'dismissal_kind', 'fielder'],
      dtype='object')

# Feature Engineering

**creating new features - 'current_score','runs_left','balls_left','wicket_left'**

In [27]:
delivery['current_score'] = delivery.groupby('match_id')['total_runs_y'].cumsum()

In [28]:
delivery['runs_left']=delivery['total_runs_x']-delivery['current_score'] + 1

In [29]:
delivery['balls_left']=126-(delivery['over']*6+delivery['ball'])

In [30]:
### wicket_left
delivery['player_dismissed'].isnull().sum() 

47118

In [31]:
delivery['player_dismissed']=delivery['player_dismissed'].fillna('0')
delivery['player_dismissed']=delivery['player_dismissed'].apply(lambda x:x if x=="0" else "1")
delivery['player_dismissed']=delivery['player_dismissed'].astype("int")
wickets = delivery.groupby('match_id')['player_dismissed'].cumsum()
delivery['wickets_left']=10-wickets
delivery.head(5)

,match_id,city,winner,total_runs_x,toss_decision,inning,batting_team,bowling_team,over,ball,...,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left
125,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,...,1,0,1,0,NaN,NaN,1,207,119,10
126,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,...,0,0,0,0,NaN,NaN,1,207,118,10
127,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,...,0,0,0,0,NaN,NaN,1,207,117,10
128,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,...,2,0,2,0,NaN,NaN,3,205,116,10
129,1,Hyderabad,Sunrisers Hyderabad,207,field,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,...,4,0,4,0,NaN,NaN,7,201,115,10


In [32]:
### current_run_rate

delivery['crr'] = (delivery['current_score']*6)/(120-delivery['balls_left'])

In [33]:
### required run rate

delivery['rrr'] = (delivery['runs_left']*6)/(delivery['balls_left'])

In [34]:
def result(row):
    return 1 if row['batting_team']==row['winner'] else 0

In [35]:
delivery['result']=delivery.apply(result,axis=1)

In [36]:
delivery.columns

Index(['match_id', 'city', 'winner', 'total_runs_x', 'toss_decision', 'inning',
       'batting_team', 'bowling_team', 'over', 'ball', 'batsman',
       'non_striker', 'bowler', 'is_super_over', 'bye_runs', 'legbye_runs',
       'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs',
       'total_runs_y', 'player_dismissed', 'dismissal_kind', 'fielder',
       'current_score', 'runs_left', 'balls_left', 'wickets_left', 'crr',
       'rrr', 'result'],
      dtype='object')

In [37]:
delivery = delivery.loc[:,['match_id','ball','city','batting_team','toss_decision','total_runs_x','bowling_team','current_score','runs_left','balls_left','wickets_left','crr','rrr','result']]

In [38]:
final_df = delivery.sample(delivery.shape[0])

In [39]:
final_df.isnull().sum()

match_id           0
ball               0
city             589
batting_team       0
toss_decision      0
total_runs_x       0
bowling_team       0
current_score      0
runs_left          0
balls_left         0
wickets_left       0
crr                0
rrr                5
result             0
dtype: int64

In [40]:
final_df.dropna(inplace=True)

In [41]:
final_df=final_df[final_df['balls_left']!=0]

In [42]:
final_df.columns

Index(['match_id', 'ball', 'city', 'batting_team', 'toss_decision',
       'total_runs_x', 'bowling_team', 'current_score', 'runs_left',
       'balls_left', 'wickets_left', 'crr', 'rrr', 'result'],
      dtype='object')

In [43]:
final_df.head(4)

,match_id,ball,city,batting_team,toss_decision,total_runs_x,bowling_team,current_score,runs_left,balls_left,wickets_left,crr,rrr,result
16241,153,1,Port Elizabeth,Royal Challengers Bangalore,bat,157,Mumbai Indians,44,114,77,7,6.139535,8.883117,0
14845,139,3,Durban,Kings XI Punjab,bat,145,Royal Challengers Bangalore,97,49,27,7,6.258065,10.888889,0
85952,7944,7,Bengaluru,Sunrisers Hyderabad,field,222,Royal Challengers Bangalore,187,36,11,8,10.293578,19.636364,0
84571,7937,6,Indore,Kings XI Punjab,field,250,Kolkata Knight Riders,129,122,42,5,9.923077,17.428571,0


In [44]:
final_df.rename(columns={'total_runs_x': 'target'}, inplace=True)
final_df.to_csv("final_df.csv", index=False)